# 04 - Model Evaluation & Anomaly DetectionHow good is the saved model really, where does it fail, and what does theIsolation Forest flag?

In [ ]:
import sysfrom pathlib import Path# Make the project importable when the notebook runs from notebooks/ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snssns.set_theme(style="whitegrid")plt.rcParams["figure.figsize"] = (12, 4)

In [ ]:
from src.data.preprocess import load_processed_datafrom src.models import predict as predictorfrom src.models.evaluate import evaluate_predictionsfrom src.anomaly.detector import AnomalyDetector, summarise_anomaliesfrom src.utils import configdf = load_processed_data()meta = predictor.get_model_metadata()print(f"Model: {meta['best_model']} | trained {meta['trained_at']}")meta["metrics"][meta["best_model"]]

## 1. Actual vs predicted on the held-out month

In [ ]:
test_start = pd.Timestamp(meta["test_period"][0])history = predictor.predict_on_history(df)test_view = history[history["timestamp"] >= test_start]metrics = evaluate_predictions(test_view["actual"], test_view["predicted"])print({k: round(v, 3) for k, v in metrics.items()})plt.figure(figsize=(13, 4))plt.plot(test_view["timestamp"], test_view["actual"], lw=1.2, label="Actual")plt.plot(test_view["timestamp"], test_view["predicted"], lw=1.2, ls="--", label="Predicted")plt.title("Held-out month: actual vs predicted active power")plt.ylabel("P (W)"); plt.legend(); plt.tight_layout(); plt.show()

## 2. Error analysisThree questions: is the model biased, are the errors concentrated at certainhours, and does error grow with load?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))axes[0].hist(test_view["error"], bins=40, color="tab:blue")axes[0].axvline(0, color="red", ls="--")axes[0].set_title(f"Error distribution (bias = {test_view['error'].mean():+.1f} W)")by_hour = test_view.assign(hour=test_view["timestamp"].dt.hour).groupby("hour")["abs_error"].mean()by_hour.plot(kind="bar", ax=axes[1], color="tab:orange")axes[1].set_title("Mean absolute error by hour")axes[2].scatter(test_view["actual"], test_view["abs_error"], s=6, alpha=0.3)axes[2].set_title("Absolute error vs actual load")axes[2].set_xlabel("Actual P (W)")plt.tight_layout(); plt.show()

Errors are largest during the evening peak and during abnormal episodes — thehardest hours to predict, and the ones a utility cares about most. This is anhonest limitation to state in a viva rather than hide.## 3. Multi-step forecast degradationRecursive forecasting feeds predictions back in as lags, so error compounds.

In [ ]:
forecast = predictor.forecast(df, horizon=24)plt.figure(figsize=(12, 4))plt.plot(df["timestamp"].tail(72), df["active_power"].tail(72), label="Measured")plt.plot(forecast["timestamp"], forecast["predicted_power_w"], "--o", ms=4, label="Forecast")plt.title("24-hour recursive forecast"); plt.ylabel("P (W)")plt.legend(); plt.tight_layout(); plt.show()print(f"Predicted energy for the next 24 h: {forecast['predicted_energy_kwh'].sum():.2f} kWh")

## 4. Baseline comparisonA forecasting model earns its place only by beating naive baselines:- **Persistence**: tomorrow's hour = this hour (`lag_1`)- **Seasonal naive**: same hour yesterday (`lag_24`)

In [ ]:
from src.features.feature_engineering import build_training_framefrom src.models.evaluate import rmse, maeX, y, names, ts = build_training_frame(df)mask = ts >= test_startbaselines = {    "Persistence (lag_1)": X.loc[mask, "lag_1"],    "Seasonal naive (lag_24)": X.loc[mask, "lag_24"],    f"{meta['best_model']}": test_view["predicted"].to_numpy(),}rows = [{"model": k, "MAE": mae(y[mask], v), "RMSE": rmse(y[mask], v)}        for k, v in baselines.items()]pd.DataFrame(rows).set_index("model").round(2)

## 5. Anomaly detectionIsolation Forest over `active_power`, `current`, `voltage`, `power_factor`.A flag means an **abnormal consumption pattern**, not a confirmed fault.

In [ ]:
detector = AnomalyDetector.load()labelled = detector.label_dataframe(df)stats = summarise_anomalies(labelled)stats

In [ ]:
normal = labelled[~labelled["is_anomaly"]]abnormal = labelled[labelled["is_anomaly"]]fig, axes = plt.subplots(1, 2, figsize=(14, 4))axes[0].plot(normal["timestamp"], normal["active_power"], lw=0.4, label="Normal")axes[0].scatter(abnormal["timestamp"], abnormal["active_power"],                color="red", s=14, marker="x", label="Abnormal")axes[0].set_title("Anomaly timeline"); axes[0].legend()axes[1].scatter(normal["voltage"], normal["current"], s=4, alpha=0.2, label="Normal")axes[1].scatter(abnormal["voltage"], abnormal["current"], s=14, color="red",                marker="x", label="Abnormal")axes[1].set_xlabel("Voltage (V)"); axes[1].set_ylabel("Current (A)")axes[1].set_title("Operating points"); axes[1].legend()plt.tight_layout(); plt.show()

In [ ]:
labelled.nsmallest(10, "anomaly_score")[    ["timestamp", "voltage", "current", "power_factor", "active_power", "anomaly_score"]]

## Conclusions- The selected model beats both naive baselines on the held-out month.- Residual error concentrates at the evening peak and during abnormal episodes.- Recursive forecasts degrade with horizon; short horizons are the trustworthy ones.- The anomaly detector isolates surges and collapses, mostly at high load.**Limitations to state plainly**1. The dataset is synthetic; real meter data is noisier and has missing periods.2. No weather or occupancy inputs, which drive real load significantly.3. Anomalies are unlabelled, so precision/recall cannot be measured.